# Group Members
* **Awais Nazeer** (406270)
* **Muhammad Musfir Baig** (409968)

# NUST Bank RAG QA System — Google Colab

**Full pipeline on Google Colab cloud:**

| Step | Script | What it does |
|------|--------|--------------|
| 0 | `inspect_data.py` | Diagnose the Excel workbook |
| 1 | `format_for_finetuning.py` | Extract QA pairs from Excel + JSON FAQ |
| 2 | `embedder.py` | Build FAISS index from QA pairs |
| 3 | `embedder_2.py` | Build Milvus Lite vector store |
| 4 | `search.py` | Similarity search (no LLM) |
| 5 | `llm.py` | Full RAG inference with Qwen via Ollama |

> **Before running:** Upload `NUST Bank-Product-Knowledge.xlsx` and `funds_transer_app_features_faq.json` when prompted in the _Project Setup_ cell.

---
## (Optional) Mount Google Drive
Run this cell if you want to save outputs to your Google Drive for persistence across sessions.

In [1]:
# OPTIONAL — uncomment to mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/LLM_Project   # change to your Drive folder if using Drive

# Default: work directly in /content (Colab's local storage)
import os
os.makedirs('/content/data', exist_ok=True)
os.chdir('/content')
print('Working directory:', os.getcwd())

Working directory: /content


---
## Step 1 — Install Python Packages
Installs all packages from `requirements.txt`. Also adds `pandas` (needed by `inspect_data.py`).

In [2]:
%%bash
# Uninstall to ensure a clean slate for the 'core' update
pip uninstall -y langchain-core

pip install -q \
    "langchain-classic>=1.0.1" \
    "langchain-core" \
    "langchain-community" \
    "langchain-ollama>=1.0.1" \
    "langchain-huggingface>=1.2.1" \
    "langchain-text-splitters>=1.1.1" \
    "pymilvus[milvus-lite]>=2.4.0" \
    "langchain-milvus>=0.1.0" \
    "sentence-transformers>=3.0.0" \
    "huggingface_hub" \
    "faiss-cpu" \
    "ollama>=0.6.1" \
    "openpyxl" \
    "pandas"

echo "✅ All packages installed. PLEASE RESTART RUNTIME NOW."

Found existing installation: langchain-core 1.2.16
Uninstalling langchain-core-1.2.16:
  Successfully uninstalled langchain-core-1.2.16
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.7/502.7 kB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 98.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.2/301.2 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.0 MB/s eta 0:00:00
✅ All packages installed. PLEASE RESTART RUNTIME NOW.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


---
## Step 2 — Install Ollama & Download Qwen Model
Installs Ollama (the local LLM server), starts it as a background process, then pulls the **Qwen 3.5 4B** model (~2.5 GB). This may take several minutes depending on Colab network speed.

In [3]:
%%bash
# Install zstd with verbose output
echo "Installing zstd..."
apt-get update && apt-get install -y zstd
echo "verifying zstd..."
which zstd && zstd --version

# Install Ollama
echo -e "\n=== Installing Ollama ==="
curl -fsSL https://ollama.com/install.sh | sh

echo -e "\n=== Verifying Ollama ==="
if command -v ollama &> /dev/null; then
    ollama --version
else
    echo "WARNING: ollama command not in PATH yet. Checking /usr/local/bin..."
    ls -la /usr/local/bin/ollama || echo "Not found in /usr/local/bin"
fi

Installing zstd...
Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.2 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,389 kB]
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:9 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,924 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:13 https://ppa.launchpadco

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [4]:
import subprocess
import time
import os

# Find ollama executable
ollama_path = None
possible_paths = [
    '/usr/local/bin/ollama',
    '/bin/ollama',
    '/usr/bin/ollama',
]

for path in possible_paths:
    if os.path.exists(path):
        ollama_path = path
        break

if not ollama_path:
    print("ERROR: ollama not found in expected paths")
    print("Please run the 'Install Ollama' cell above first and check for errors.")
    raise FileNotFoundError("ollama executable not found")

print(f'Found ollama at: {ollama_path}')

# Start the Ollama server as a background daemon
print('Starting Ollama server...')
ollama_proc = subprocess.Popen(
    [ollama_path, 'serve'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(8)  # give the server time to start
print(f'Ollama server started (PID {ollama_proc.pid})')
print('Waiting for server to be ready...')

Found ollama at: /usr/local/bin/ollama
Starting Ollama server...
Ollama server started (PID 2753)
Waiting for server to be ready...


In [5]:
%%bash
# Set PATH to include /usr/local/bin where ollama is installed
export PATH="/usr/local/bin:$PATH"

echo "=== Pulling Qwen 3.5 Model (this may take several minutes) ==="

# Download the Qwen model — adjust the model name if needed
if command -v ollama &> /dev/null; then
    ollama pull qwen3.5:4b
    echo -e "\n=== Model download complete ==="
    ollama list
else
    echo "ERROR: ollama command not found. Please check the installation cell above."
    exit 1
fi

=== Pulling Qwen 3.5 Model (this may take several minutes) ===

=== Model download complete ===
NAME          ID              SIZE      MODIFIED               
qwen3.5:4b    2a654d98e6fb    3.4 GB    Less than a second ago    


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest 
pulling 81fb60c7daa8:   1% ▕                  ▏  21 MB/3.4 GB                  pulling manifest 
pulling 81fb60c7daa8:   3% ▕                  ▏ 101 MB/3.4 GB                  pulling manifest 
pulling 81fb60c7daa8:   4% ▕                  ▏ 152 MB/3.4 GB                  pulling manifest 
pulling 81fb60c7daa8:   5% ▕                  ▏ 173 MB/3.4 GB                  pulling manifest 
pulling 81fb60c7daa8:   6% ▕█                 ▏ 218 MB/3.4 GB                  pulling manifest 
pulling 81fb60c7daa8:   8% ▕█                 ▏ 262 MB/3.4 GB                  pulling manifest 
pulling 81fb60c7daa8:   8% ▕█                 ▏ 283 MB/3.4 GB                  pulling manifest 
pulling 81fb60c7daa8:  10% ▕█                 ▏ 329 MB/3.4 GB                  pulling manifest 
pulling 81fb60c7

---
## Step 3 — Upload Project Data Files

**Two ways to upload:**

### Option A: Upload button (recommended)
Run the cell below and click "Choose Files" to select:
- `NUST Bank-Product-Knowledge.xlsx`
- `funds_transer_app_features_faq.json`

### Option B: Manual upload (if button doesn't work)
1. Click the **Files** icon on the left sidebar in Colab
2. Drag and drop both files into the file explorer
3. They'll appear in `/content/` automatically
4. Run the verification cell to confirm

In [6]:
from google.colab import files
import shutil
import os
import time

print("=" * 70)
print("FILE UPLOAD - Option 1: Use the button below")
print("=" * 70)
print("\nA file upload dialog will appear below.")
print("Click 'Choose Files' and select both:")
print("  1. NUST Bank-Product-Knowledge.xlsx")
print("  2. funds_transer_app_features_faq.json")
print("\nWaiting for files (timeout after 2 minutes)...\n")

try:
    # Set a timeout for file selection
    uploaded = files.upload()

    if uploaded:
        for fname in uploaded.keys():
            src = fname
            dst = f'/content/{fname}'
            if os.path.abspath(src) != os.path.abspath(dst):
                shutil.move(src, dst)
            print(f'✓ Uploaded: {dst}')

        # Verify
        required = ['NUST Bank-Product-Knowledge.xlsx', 'funds_transer_app_features_faq.json']
        print("\n" + "=" * 70)
        print("VERIFICATION")
        print("=" * 70)
        for f in required:
            path = f'/content/{f}'
            if os.path.exists(path):
                size = os.path.getsize(path) / (1024 * 1024)
                print(f'✓ [{f}] {size:.1f} MB')
            else:
                print(f'✗ [MISSING] {f}')
    else:
        print("No files uploaded.")

except Exception as e:
    print(f"Upload interrupted or timed out: {e}")
    print("\nIf upload didn't work, try Option 2 below.")

FILE UPLOAD - Option 1: Use the button below

A file upload dialog will appear below.
Click 'Choose Files' and select both:
  1. NUST Bank-Product-Knowledge.xlsx
  2. funds_transer_app_features_faq.json

Waiting for files (timeout after 2 minutes)...



Saving funds_transer_app_features_faq.json to funds_transer_app_features_faq.json
Saving NUST Bank-Product-Knowledge.xlsx to NUST Bank-Product-Knowledge.xlsx
✓ Uploaded: /content/funds_transer_app_features_faq.json
✓ Uploaded: /content/NUST Bank-Product-Knowledge.xlsx

VERIFICATION
✓ [NUST Bank-Product-Knowledge.xlsx] 1.8 MB
✓ [funds_transer_app_features_faq.json] 0.0 MB


---
## Step 4 — Inspect Excel Workbook (`inspect_data.py`)
Prints a diagnostic summary for every sheet: dimensions, column types, missing values, and preview rows.

In [7]:
import os

print("=" * 70)
print("CHECKING UPLOADED FILES")
print("=" * 70)

required_files = {
    '/content/NUST Bank-Product-Knowledge.xlsx': 'Excel workbook',
    '/content/funds_transer_app_features_faq.json': 'JSON FAQ'
}

all_present = True
for fpath, desc in required_files.items():
    if os.path.exists(fpath):
        size_mb = os.path.getsize(fpath) / (1024 * 1024)
        print(f'✓ {desc:30s} {size_mb:>8.1f} MB  {fpath}')
    else:
        print(f'✗ {desc:30s} [NOT FOUND]  {fpath}')
        all_present = False

print("=" * 70)
if all_present:
    print("✓ All files ready. Proceed to the next step.")
else:
    print("✗ Some files are missing. Please upload them first.")
    print("\nFor manual upload in Colab:")
    print("  1. Click the 'Files' icon (folder) on the left sidebar")
    print("  2. Drag-and-drop your files into the file explorer")
    print("  3. Files will appear in /content/ automatically")
    print("  4. Re-run this cell to verify")
print("=" * 70)

"""
Excel Workbook Inspector  —  inspect_data.py
Iterates over every sheet in the NUST Bank knowledge-base Excel file and
prints a diagnostic summary: shape, column names, data types, missing-value
counts, unique-value counts, and preview rows.
"""

import pandas as pd

if all_present:
    WORKBOOK = '/content/NUST Bank-Product-Knowledge.xlsx'


    def inspect_sheet(xls: pd.ExcelFile, name: str):
        """Print a diagnostic block for a single sheet."""
        df = pd.read_excel(xls, sheet_name=name)

        separator = '-' * 76
        print(f'\n{separator}')
        print(f'Sheet: {name}')
        print(separator)
        print(f'  Dimensions : {df.shape[0]} rows x {df.shape[1]} columns')
        print(f'  Columns    : {list(df.columns)}\n')

        print('  Column types:')
        for col in df.columns:
            print(f'    {col:40s} => {df[col].dtype}')

        print('\n  Missing values:')
        for col in df.columns:
            nulls = df[col].isna().sum()
            ratio = nulls / len(df) * 100 if len(df) else 0
            print(f'    {col:40s} => {nulls:>5}  ({ratio:.1f}%)')

        print('\n  Distinct values:')
        for col in df.columns:
            print(f'    {col:40s} => {df[col].nunique()}')

        display_opts = {'display.max_columns': None, 'display.width': 200, 'display.max_colwidth': 60}
        with pd.option_context(*[item for pair in display_opts.items() for item in pair]):
            print('\n  Head (5 rows):')
            print(df.head().to_string(index=False))
            print('\n  Tail (2 rows):')
            print(df.tail(2).to_string(index=False))


    def inspect_main():
        xls = pd.ExcelFile(WORKBOOK, engine='openpyxl')

        banner = '=' * 76
        print(f'\n{banner}')
        print(f'Workbook  : {os.path.basename(WORKBOOK)}')
        print(f'Sheets    : {len(xls.sheet_names)}')
        print(f'Names     : {xls.sheet_names}')
        print(banner)

        for sheet_name in xls.sheet_names:
            inspect_sheet(xls, sheet_name)

        print(f'\n{banner}')
        print('Inspection finished.')
        print(banner)


    print('\n\nStarting workbook inspection...\n')
    inspect_main()
else:
    print("\n[SKIPPED] Workbook inspection — files not ready")

CHECKING UPLOADED FILES
✓ Excel workbook                      1.8 MB  /content/NUST Bank-Product-Knowledge.xlsx
✓ JSON FAQ                            0.0 MB  /content/funds_transer_app_features_faq.json
✓ All files ready. Proceed to the next step.


Starting workbook inspection...


Workbook  : NUST Bank-Product-Knowledge.xlsx
Sheets    : 36
Names     : ['Main', 'Rate Sheet July 1 2024', 'LCA', 'NAA', 'NWA', 'PWRA', 'RDA', 'VPCA', 'VP-BA', 'VPBA', 'NSDA', 'PLS', 'CDA', 'NMA', 'NADA', 'NADRA', 'NUST4Car', 'ESFCA', 'NFDA', 'NSA', 'PF', 'NMC', 'NMF', 'NSF', 'NIF', 'NUF', 'NFMF', 'NFBF', 'PMYB &ALS', 'NRF', 'NHF', 'Nust Life', 'EFU Life', 'Jubilee Life ', 'HOME REMITTANCE', 'Sheet1']

----------------------------------------------------------------------------
Sheet: Main
----------------------------------------------------------------------------
  Dimensions : 18 rows x 6 columns
  Columns    : ['Unnamed: 0', 'NUST BANK PRODUCTS (CONVENTIONAL)\n(Click on any product below)', 'Unnamed: 2'

---
## Step 5 — Extract QA Pairs (`format_for_finetuning.py`)
Parses all 34+ sheets, detects question/answer rows via heuristics, merges the supplementary JSON FAQ, and writes three output files:
- `data/all_qa_pairs.json` — raw QA pairs
- `data/finetuning_data.jsonl` — Alpaca instruction format
- `data/finetuning_data_chat.jsonl` — OpenAI chat-completion format

In [8]:
"""
Excel QA Extractor & Fine-tuning Formatter  —  data/format_for_finetuning.py
Parses the NUST Bank product knowledge Excel workbook, detects question /
answer rows using heuristics, and writes three output files.
"""

import json
import os
import re
from collections import Counter

import openpyxl

# ── Path configuration for Colab ─────────────────────────────────────────
WORKBOOK_PATH      = '/content/NUST Bank-Product-Knowledge.xlsx'
OUT_ALL_QA_JSON    = '/content/data/all_qa_pairs.json'
OUT_INSTRUCT_JSONL = '/content/data/finetuning_data.jsonl'
OUT_CHAT_JSONL     = '/content/data/finetuning_data_chat.jsonl'

IGNORED_SHEETS = {'Main', 'Rate Sheet July 1 2024', 'Sheet1'}

ASSISTANT_PERSONA = (
    'You are a helpful customer support assistant for NUST Bank. '
    'Answer the customer\'s question accurately and concisely using '
    'the bank\'s product knowledge.'
)


# ── Text utilities ───────────────────────────────────────────────────────

def tidy(text: str) -> str:
    """Normalise whitespace and remove tabs while preserving paragraph breaks."""
    if not text:
        return ''
    text = text.strip().replace('\t', ' ')
    text = re.sub(r'[^\S\n]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()


def looks_like_question(text: str) -> bool:
    """Heuristic to decide whether *text* is a question row."""
    s = text.strip()
    if not s:
        return False
    if s.endswith('?'):
        return True
    QUESTION_PREFIXES = (
        'what', 'how', 'is ', 'is\n', 'can ', 'can\n', 'does', 'do ',
        'are ', 'which', 'who', 'where', 'when', 'why',
        'i would like to', 'i want to', 'please tell',
        '1.', '1 .', '1-',
    )
    low = s.lower()
    if any(low.startswith(pfx) for pfx in QUESTION_PREFIXES):
        return True
    if '?' in s[:80]:
        return True
    return False


# ── Sheet-level extraction ───────────────────────────────────────────────

def _read_product_title(ws) -> str:
    for row in ws.iter_rows(min_row=1, max_row=1, values_only=False):
        for cell in row:
            if cell.value is None:
                continue
            candidate = str(cell.value).strip()
            if candidate.startswith('=') or candidate.lower() == 'main':
                continue
            if candidate:
                return candidate
    return ''


def _collect_row_text(ws) -> dict:
    mapping = {}
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row, values_only=False):
        parts = []
        for cell in row:
            if cell.value is None:
                continue
            val = str(cell.value).strip()
            if not val or val.startswith('=') or val.lower() == 'main':
                continue
            parts.append(val)
        if parts:
            joined = ' | '.join(parts) if len(parts) > 1 else parts[0]
            row_num = row.row if hasattr(row, 'row') else row[0].row
            mapping[row_num] = joined
    return mapping


def extract_pairs(ws, sheet_label: str) -> list:
    product = _read_product_title(ws) or sheet_label
    row_texts = _collect_row_text(ws)
    if not row_texts:
        return []

    ordered = sorted(row_texts.items())
    offset = 1 if ordered and ordered[0][1].strip() == product.strip() else 0

    pairs = []
    pending_q = None
    pending_a_lines = []

    def _flush():
        nonlocal pending_q, pending_a_lines
        if pending_q and pending_a_lines:
            combined = tidy('\n'.join(pending_a_lines))
            if combined:
                pairs.append({
                    'question': tidy(pending_q),
                    'answer':   combined,
                    'product':  product,
                    'sheet':    sheet_label,
                })
        pending_q = None
        pending_a_lines.clear()

    for _, text in ordered[offset:]:
        text = tidy(text)
        if not text:
            continue
        if looks_like_question(text):
            _flush()
            pending_q = text
        elif pending_q is not None:
            pending_a_lines.append(text)

    _flush()
    return pairs


# ── JSON FAQ supplement ──────────────────────────────────────────────────

def _ingest_json_faq(path: str) -> list:
    if not os.path.exists(path):
        return []
    with open(path, 'r', encoding='utf-8') as fh:
        blob = json.load(fh)
    extra = []
    for cat in blob.get('categories', []):
        label = cat.get('category', 'General')
        for item in cat.get('questions', []):
            extra.append({
                'question': tidy(item['question']),
                'answer':   tidy(item['answer']),
                'product':  f'Mobile App - {label}',
                'sheet':    'funds_transfer_app_features_faq.json',
            })
    return extra


# ── Output writers ───────────────────────────────────────────────────────

def _write_all_qa_json(qa_list: list, dest: str):
    with open(dest, 'w', encoding='utf-8') as fh:
        json.dump(qa_list, fh, indent=2, ensure_ascii=False)
    print(f'  -> {dest}  ({len(qa_list)} pairs)')


def _write_instruct_jsonl(qa_list: list, dest: str):
    with open(dest, 'w', encoding='utf-8') as fh:
        for qa in qa_list:
            fh.write(json.dumps({
                'instruction': qa['question'],
                'input':       '',
                'output':      qa['answer'],
                'product':     qa['product'],
            }, ensure_ascii=False) + '\n')
    print(f'  -> {dest}')


def _write_chat_jsonl(qa_list: list, dest: str):
    with open(dest, 'w', encoding='utf-8') as fh:
        for qa in qa_list:
            fh.write(json.dumps({
                'messages': [
                    {'role': 'system',    'content': ASSISTANT_PERSONA},
                    {'role': 'user',      'content': qa['question']},
                    {'role': 'assistant', 'content': qa['answer']},
                ],
            }, ensure_ascii=False) + '\n')
    print(f'  -> {dest}')


# ── Report ───────────────────────────────────────────────────────────────

def _print_report(qa_list: list):
    freq = Counter(qa['product'] for qa in qa_list)
    bar = '-' * 58
    print(f'\n{bar}')
    print(f'{"Product":<44} {"Count":>10}')
    print(bar)
    for name, cnt in freq.most_common():
        print(f'  {name:<42} {cnt:>10}')
    print(bar)
    print(f'  {"TOTAL":<42} {len(qa_list):>10}')
    print(f'\n{"=" * 58}')
    print('Sample entries (first 3):')
    print('=' * 58)
    for i, qa in enumerate(qa_list[:3], start=1):
        print(f'\n  [{i}] Product: {qa["product"]}')
        print(f'      Q: {qa["question"][:120]}')
        print(f'      A: {qa["answer"][:200]}')


# ── Run ──────────────────────────────────────────────────────────────────

def format_main():
    # Ensure the output directory exists
    output_dir = os.path.dirname(OUT_ALL_QA_JSON)
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        print(f"Created directory: {output_dir}")
    # ---------------------------
    wb = openpyxl.load_workbook(WORKBOOK_PATH, data_only=True)

    all_pairs = []
    skipped = []

    for name in wb.sheetnames:
        if name in IGNORED_SHEETS:
            skipped.append(name)
            continue
        sheet_pairs = extract_pairs(wb[name], name)
        all_pairs.extend(sheet_pairs)
        print(f'  + {name:25s}  {len(sheet_pairs):>3} pairs')

    print(f'\nIgnored sheets: {skipped}')
    print(f'Extracted from workbook: {len(all_pairs)} pairs')

    faq_path = '/content/funds_transer_app_features_faq.json'
    supplementary = _ingest_json_faq(faq_path)
    if supplementary:
        all_pairs.extend(supplementary)
        print(f'  + {"JSON FAQ":25s}  {len(supplementary):>3} pairs')
        print(f'Combined total: {len(all_pairs)} pairs')

    _write_all_qa_json(all_pairs, OUT_ALL_QA_JSON)
    _write_instruct_jsonl(all_pairs, OUT_INSTRUCT_JSONL)
    _write_chat_jsonl(all_pairs, OUT_CHAT_JSONL)
    _print_report(all_pairs)


format_main()

  + LCA                          7 pairs
  + NAA                          5 pairs
  + NWA                          5 pairs
  + PWRA                         7 pairs
  + RDA                          7 pairs
  + VPCA                         4 pairs
  + VP-BA                        5 pairs
  + VPBA                         5 pairs
  + NSDA                         5 pairs
  + PLS                          4 pairs
  + CDA                          4 pairs
  + NMA                          5 pairs
  + NADA                         6 pairs
  + NADRA                        5 pairs
  + NUST4Car                    19 pairs
  + ESFCA                        5 pairs
  + NFDA                         5 pairs
  + NSA                         15 pairs
  + PF                          13 pairs
  + NMC                         13 pairs
  + NMF                         20 pairs
  + NSF                         15 pairs
  + NIF                         18 pairs
  + NUF                         12 pairs
  + NFMF        

---
## Step 6 — Stage 1: Build FAISS Index (`embedder.py`)
Reads `data/all_qa_pairs.json`, sanitises text, encodes every QA pair with `all-MiniLM-L6-v2` (384-dim), and writes:
- `data/faiss_index.bin`
- `data/chunk_metadata.json`

In [ ]:
# !pip install faiss-cpu sentence-transformers

In [9]:
"""
Stage-1 Embedding Pipeline  —  embedder.py
Reads the cleaned QA-pair JSON, normalises every entry, converts each pair
into an embeddable text chunk, generates dense vectors with a
SentenceTransformer encoder, and persists a FAISS index alongside the
corresponding chunk metadata.
"""

import json
import re
import pathlib

import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

# ── Configuration ────────────────────────────────────────────────────────
QA_SOURCE_FILE   = pathlib.Path('/content/data/all_qa_pairs.json')
INDEX_DEST       = pathlib.Path('/content/data/faiss_index.bin')
METADATA_DEST    = pathlib.Path('/content/data/chunk_metadata.json')
ENCODER_MODEL_ID = 'all-MiniLM-L6-v2'


# ── Text pre-processing ─────────────────────────────────────────────────

def _sanitise(raw_text: str) -> str:
    """Strip pipe-table artefacts, collapse whitespace, remove bullets."""
    result = re.sub(r'\s*\|\s*', ' ', raw_text)
    result = re.sub(r'\s+', ' ', result)
    result = re.sub(r'^[^\w\s]+\s*', '', result, flags=re.MULTILINE)
    return result.strip()


# ── Pipeline stages ──────────────────────────────────────────────────────

def ingest_qa_pairs(src: pathlib.Path) -> list:
    """Read QA pairs from *src*, discarding entries with blank fields."""
    with open(src, 'r', encoding='utf-8') as fh:
        raw = json.load(fh)
    filtered = [
        rec for rec in raw
        if rec.get('question', '').strip() and rec.get('answer', '').strip()
    ]
    print(f'[load] kept {len(filtered)} of {len(raw)} entries')
    return filtered


def assemble_chunks(qa_records: list) -> list:
    """Turn each QA record into a single text chunk with metadata."""
    output = []
    for pos, rec in enumerate(qa_records):
        q = _sanitise(rec['question'])
        a = _sanitise(rec['answer'])
        output.append({
            'id':       pos,
            'text':     f'Question: {q}\nAnswer: {a}',
            'question': q,
            'answer':   a,
            'product':  rec.get('product', 'Unknown'),
            'sheet':    rec.get('sheet', 'Unknown'),
        })
    print(f'[chunk] assembled {len(output)} passages')
    return output


def encode_passages(chunks: list, model_id: str = ENCODER_MODEL_ID):
    """Vectorise chunk texts and return (matrix, encoder)."""
    print(f'[encode] initialising \'{model_id}\' ...')
    encoder = SentenceTransformer(model_id)
    passages = [c['text'] for c in chunks]
    print(f'[encode] processing {len(passages)} passages ...')
    matrix = encoder.encode(passages, show_progress_bar=True, convert_to_numpy=True)
    print(f'[encode] matrix shape: {matrix.shape}')
    return matrix, encoder


def write_faiss_artefacts(
    matrix: np.ndarray,
    chunks: list,
    idx_path: pathlib.Path = INDEX_DEST,
    meta_path: pathlib.Path = METADATA_DEST,
):
    """Construct a flat-L2 FAISS index and persist everything to disk."""
    dim = matrix.shape[1]
    store = faiss.IndexFlatL2(dim)
    store.add(matrix)

    faiss.write_index(store, str(idx_path))
    print(f'[write] index   -> {idx_path}')

    with open(meta_path, 'w', encoding='utf-8') as fh:
        json.dump(chunks, fh, ensure_ascii=False, indent=4)
    print(f'[write] metadata -> {meta_path}')
    return store


# ── Orchestrator ─────────────────────────────────────────────────────────

def embedder_run():
    records   = ingest_qa_pairs(QA_SOURCE_FILE)
    chunks    = assemble_chunks(records)
    matrix, _ = encode_passages(chunks)
    store     = write_faiss_artefacts(matrix, chunks)
    print(
        f'\n[summary] chunks={len(chunks)}  dim={matrix.shape[1]}  '
        f'vectors={matrix.shape}  stored={store.ntotal}'
    )


embedder_run()

[load] kept 317 of 317 entries
[chunk] assembled 317 passages
[encode] initialising 'all-MiniLM-L6-v2' ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[encode] processing 317 passages ...


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

[encode] matrix shape: (317, 384)
[write] index   -> /content/data/faiss_index.bin
[write] metadata -> /content/data/chunk_metadata.json

[summary] chunks=317  dim=384  vectors=(317, 384)  stored=317


---
## Step 7 — Stage 2: Build Milvus Vector Store (`embedder_2.py`)
Reads `data/chunk_metadata.json`, wraps each chunk as a LangChain Document, embeds with HuggingFace, and upserts into a **Milvus Lite** collection (single `.db` file — no Docker needed).

In [10]:
"""
Stage-2: Milvus Vector Store Builder  —  embedder_2.py
Reads chunk metadata produced by the stage-1 embedder, wraps each chunk as
a LangChain Document, computes embeddings via HuggingFace, and upserts
everything into a Milvus Lite collection stored as a local .db file.
"""

import json
import pathlib

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_milvus import Milvus

# ── Configuration ─────────────────────────────────────────────────────────
CHUNK_META_PATH = pathlib.Path('/content/data/chunk_metadata.json')
MILVUS_DB_PATH  = '/content/data/milvus_bank.db'
COLLECTION_NAME = 'bank_knowledge'
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'


# ── Helpers ───────────────────────────────────────────────────────────────

def _load_chunks(path: pathlib.Path) -> list:
    with open(path, 'r', encoding='utf-8') as fh:
        return json.load(fh)


def _to_documents(chunks: list) -> list:
    """Wrap each chunk dict in a LangChain Document."""
    return [
        Document(
            page_content=ch['text'],
            metadata={
                'chunk_id': ch['id'],
                'question': ch['question'],
                'answer':   ch['answer'],
                'product':  ch.get('product', 'Unknown'),
                'sheet':    ch.get('sheet',   'Unknown'),
            },
        )
        for ch in chunks
    ]


# ── Main builder ──────────────────────────────────────────────────────────

def build_milvus_store():
    chunks = _load_chunks(CHUNK_META_PATH)
    docs   = _to_documents(chunks)
    print(f'[milvus] {len(docs)} documents ready for insertion')

    embedder = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
    print(f'[milvus] embedding model \'{EMBEDDING_MODEL}\' loaded')

    vec_store = Milvus.from_documents(
        docs,
        embedder,
        connection_args={'uri': MILVUS_DB_PATH},
        collection_name=COLLECTION_NAME,
        drop_old=True,
    )

    total = vec_store.col.num_entities
    print(f'[milvus] persisted {total} vectors  ->  {MILVUS_DB_PATH}')
    return vec_store


milvus_store = build_milvus_store()

[milvus] 317 documents ready for insertion


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[milvus] embedding model 'all-MiniLM-L6-v2' loaded
[milvus] persisted 317 vectors  ->  /content/data/milvus_bank.db


---
## Step 8 — Similarity Search (`search.py`)
Pure vector similarity search over the Milvus collection — **no LLM involved**. Useful for validating that embeddings are working correctly.

Edit `SEARCH_QUERIES` below to test your own questions.

In [11]:
"""
Standalone Similarity Search  —  search.py
Loads the Milvus Lite collection and runs similarity_search_with_score
for a list of example queries.
"""

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_milvus import Milvus

# ── Settings ───────────────────────────────────────────────────────────
MILVUS_DB_PATH  = '/content/data/milvus_bank.db'
COLLECTION_NAME = 'bank_knowledge'
MODEL_ID        = 'all-MiniLM-L6-v2'
DEFAULT_K       = 3

# ── Queries to run ─────────────────────────────────────────────────────
SEARCH_QUERIES = [
    'how do i open an account?',
    'what are the loan interest rates?',
    'how to transfer money online?',
]


def _bootstrap():
    """Load the embedding model and connect to the Milvus collection."""
    embed_fn = HuggingFaceEmbeddings(model_name=MODEL_ID)
    store = Milvus(
        embed_fn,
        connection_args={'uri': MILVUS_DB_PATH},
        collection_name=COLLECTION_NAME,
    )
    return store


def find_similar(query: str, store: Milvus, k: int = DEFAULT_K):
    """Search *store* for the top-k passages closest to *query*."""
    hits = store.similarity_search_with_score(query, k=k)
    for rank, (doc, score) in enumerate(hits, start=1):
        meta = doc.metadata
        print(f'\n--- Result {rank}  (score: {score:.4f}) ---')
        print(f'  Product : {meta.get("product", "N/A")}')
        print(f'  Q : {meta.get("question", "")}')
        print(f'  A : {meta.get("answer", "")[:200]}...')


print('Loading Milvus collection and embedding model ...')
search_store = _bootstrap()
print('Ready.\n')

for q in SEARCH_QUERIES:
    print(f'\n{"=" * 60}')
    print(f'Search: {q}')
    print('=' * 60)
    find_similar(q, search_store)

Loading Milvus collection and embedding model ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Ready.


Search: how do i open an account?

--- Result 1  (score: 0.9139) ---
  Product : NUST Sahar Accounts
  Q : Can we open this account online?
  A : Digital account opening facility available with account opening within 48 hours...

--- Result 2  (score: 0.9228) ---
  Product : Value Plus Current Account
  Q : I would like to inquire about opening a current account for Individuals with your Bank. Please tell me what options I have ?
  A : NUST Value Plus Current Account is specially designed for individuals to cater their financial needs....

--- Result 3  (score: 1.0186) ---
  Product : Little Champs Account
  Q : I would like to open an account with my son, do u have any product for kids?
  A : Yes our product is Little Champs Account. It is designed specifically for minors (individuals below the age of 18 years). A child requires the help of a parental/legal guardian to open this account an...

Search: what are the loan interest rates?

--- Result 1  (score: 0.9384) ---
  Prod

---
## Step 9 — RAG Inference (`llm.py`)
Connects the Milvus retriever to **Qwen 3.5 (4B)** running in Ollama, assembles a `RetrievalQA` chain with top-3 context chunks, and generates a final answer.

Edit `TEST_QUERIES` below to ask your own questions.

In [ ]:
#!pip install langchain-ollama langchain-classic

In [16]:
import subprocess
import time
import requests

# 1. Kill any existing Ollama processes
!pkill ollama

# 2. Start Ollama in the background
with open("ollama_logs.txt", "w") as f:
    subprocess.Popen(["ollama", "serve"], stdout=f, stderr=f)

# 3. Wait for the server to wake up
print("Waiting for Ollama to wake up...")
for _ in range(10):
    try:
        response = requests.get("http://localhost:11434/api/tags")
        if response.status_code == 200:
            print("✅ Ollama Server is UP!")
            break
    except:
        time.sleep(2)
else:
    print("❌ Server timed out. Check ollama_logs.txt")

# 4. Ensure the model is available
print("Ensuring model is available...")
!ollama pull qwen3.5:4b

# 5. NEW: Warm up the model (Forces it into memory)
print("Warming up Qwen (loading into RAM/GPU)...")
!ollama run qwen3.5:4b "hi" # This sends a tiny request to trigger the load
print("🚀 Model is hot and ready!")

Waiting for Ollama to wake up...
✅ Ollama Server is UP!
Ensuring model is available...

Warming up Qwen (loading into RAM/GPU)...
Thinking...
Thinking Process:

1.  **Analyze the Request:**
    *   Input: "hi"
    *   Intent: Greeting, starting a conversation.
    *   Tone: Friendly, helpful, casual.

2.  **Determine the appropriate response:**
    *   Acknowledge the greeting.
    *   Offer assistance.
    *   Keep it open-ended to invite further input.

3.  **Draft potential responses:**
    *   "Hello! How can I help you today?" (Classic, safe)
    *   "Hey there! What's on your mind?" (Friendly)
    *   "Hi! How are you doing? Anything I can assist you with?" (Polite, helpful)
    *   "Hello! 👋 Ready to chat or need some help?" (Visual + text)

4.  **Select the best option:**
    *   Option 3 is balanced: friendly, polite, and clear about capabilities.
    *   Add an emoji for warmth (common in modern AI interactions).

5.  **Final Polish:**
    *   "Hi there! 👋 How can I help you 

In [20]:
# 1. Kill any process that might be locking the Milvus file
!fuser -k /content/data/milvus_bank.db

# 2. Force-clear the python cache
import gc
gc.collect()

7950

In [23]:
import subprocess
import time
import requests

# 1. Kill any zombie Ollama processes
!pkill ollama

# 2. Start the server and wait for it to be ready
print("Starting Ollama server...")
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# 3. Wait and check connectivity
for i in range(10):
    try:
        response = requests.get("http://localhost:11434/api/tags")
        if response.status_code == 200:
            print(f"✅ Ollama server is alive (Attempt {i+1})")
            break
    except:
        time.sleep(3)
else:
    print("❌ Failed to start Ollama server.")

# 4. Ensure the model is pulled
!ollama pull qwen3.5:4b
print("🚀 Server and Model are ready. Now run your Stage 3 cell!")

Starting Ollama server...
✅ Ollama server is alive (Attempt 2)

🚀 Server and Model are ready. Now run your Stage 3 cell!


In [24]:
import json
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_ollama import OllamaLLM
from langchain_classic.chains import RetrievalQA

# ── Configuration ───────────────────────────────────────────────────────
QA_SOURCE = '/content/data/all_qa_pairs.json'
GENERATIVE_MODEL = 'qwen3.5:4b'
EMBEDDING_MODEL  = 'all-MiniLM-L6-v2'

print("Step 1: Loading QA Pairs from JSON...")
with open(QA_SOURCE, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Convert your JSON into LangChain Documents
documents = [
    Document(
        page_content=f"Question: {item['question']}\nAnswer: {item['answer']}",
        metadata={"product": item.get('product', 'General')}
    ) for item in raw_data
]
print(f"✅ Loaded {len(documents)} knowledge chunks.")

print("\nStep 2: Building FAISS Index (In-Memory)...")
embed_fn = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
# This builds the index instantly in RAM - no file locking issues!
vectorstore = FAISS.from_documents(documents, embed_fn)
print("✅ FAISS Index ready.")

print("\nStep 3: Initializing Ollama LLM...")
generator = OllamaLLM(model=GENERATIVE_MODEL, timeout=120)
print("✅ LLM initialized.")

print("\nStep 4: Building RetrievalQA Chain...")
qa_chain = RetrievalQA.from_chain_type(
    llm=generator,
    chain_type='stuff',
    retriever=vectorstore.as_retriever(search_kwargs={'k': 3}),
    verbose=True
)
print("🚀 SYSTEM READY.\n")

# ── Test Query ──────────────────────────────────────────────────────────
query = "how do i open an account?"
print(f"Question: {query}")
result = qa_chain.invoke({"query": query})
print("\nAnswer:", result['result'])

Step 1: Loading QA Pairs from JSON...
✅ Loaded 317 knowledge chunks.

Step 2: Building FAISS Index (In-Memory)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ FAISS Index ready.

Step 3: Initializing Ollama LLM...
✅ LLM initialized.

Step 4: Building RetrievalQA Chain...
🚀 SYSTEM READY.

Question: how do i open an account?


> Entering new RetrievalQA chain...

> Finished chain.

Answer: Based on the information provided, there is a digital account opening facility available with account opening within 48 hours. 

Depending on your needs:
*   **For Individuals:** You may consider the NUST Value Plus Current Account.
*   **For Minors (below age of 18):** You can open the Little Champs Account, but parental or legal guardian assistance is required to open the account and avail its facilities.


---
## Step 10 — Interactive QA (optional)
Type any question and get a live RAG-generated answer. Assumes `qa_chain` is already initialised in the cell above.

In [ ]:
# Type your own question here and run the cell
MY_QUESTION = 'What is the minimum balance required for a savings account?'

result = ask(MY_QUESTION, qa_chain)
print('Q:', MY_QUESTION)
print('A:', result.get('result', result))



> Entering new RetrievalQA chain...


---
## Step 11 — Download Output Files (optional)
Download the generated data files back to your local machine.

In [ ]:
from google.colab import files
import os

outputs = [
    '/content/data/all_qa_pairs.json',
    '/content/data/finetuning_data.jsonl',
    '/content/data/finetuning_data_chat.jsonl',
    '/content/data/chunk_metadata.json',
    '/content/data/faiss_index.bin',
    '/content/data/milvus_bank.db',
]

for path in outputs:
    if os.path.exists(path):
        print(f'Downloading: {path}')
        files.download(path)
    else:
        print(f'[SKIP — not found] {path}')